In [27]:
import threading
import numpy as np
import time
from PIL import Image
from constants import PGM, IMAGE_NAME
import multiprocessing
from dataclasses import dataclass
from typing import Any

In [28]:
def image_negative_worker(start, end, result_lock, new_image, image):
    """Generate a negative for the image chunk of the image"""
    image_chunk = image[start:end].copy()
    new_image_chunk = 255 - image_chunk
    # Need lock when updating shared result
    with result_lock:
        new_image[start:end] += new_image_chunk


def threshold_with_slicing_worker(start, end, result_lock, new_image, image, limits):
    """Generate a negative for the image chunk of the image"""
    a, b = limits
    image_chunk = image[start:end].copy()
    mask = (image_chunk <= a) | (image_chunk >= b)
    new_image_chunk = np.where(mask, 255, image_chunk)

    # Need lock when updating shared result
    with result_lock:
        new_image[start:end] = new_image_chunk

In [29]:
def calculate_image_threads(height, min_rows_per_thread=50, max_threads=16):
    """
    Calculate threads for image processing

    Args:
        height: Image height in pixels
        min_rows_per_thread: Minimum rows to avoid overhead
        max_threads: Upper limit
    """
    cpu_cores = multiprocessing.cpu_count()

    # Maximum threads possible based on rows
    max_threads_by_rows = max(1, height // min_rows_per_thread)

    # Optimal threads (can't have more threads than rows)
    optimal = min(cpu_cores, max_threads_by_rows, max_threads)

    # Keep at least one CPU core free when all cores would be used
    if optimal == cpu_cores and optimal > 1:
        optimal -= 1

    # Ensure each thread gets reasonable work
    if optimal > 1:
        rows_per_thread = height // optimal
        if rows_per_thread < min_rows_per_thread:
            optimal = max(1, height // min_rows_per_thread)
            if optimal == cpu_cores and optimal > 1:
                optimal -= 1

    return optimal

In [30]:
def get_image_thread_results(image_path=IMAGE_NAME):
    with Image.open(image_path) as img:
        arr = np.array(img)
        data = arr.tobytes()
        metadata = PGM(img.height, img.width, int(arr.max()), data)
    threads = calculate_image_threads(metadata.height)
    return {
        "metadata": metadata,
        "threads": threads,
    }


def get_image_slice_values(image_path=IMAGE_NAME):
    results = get_image_thread_results(image_path)
    metadata = results["metadata"]

    width, height = metadata.width, metadata.height
    image = np.frombuffer(metadata.data, dtype=np.uint8).reshape(height, width, 3)

    num_threads = results["threads"]
    rows_per_thread = height // num_threads

    slices = []
    for i in range(num_threads):
        start = i * rows_per_thread
        end = (i + 1) * rows_per_thread if i < num_threads - 1 else height
        slices.append([start, end]) # slices

    return {
        "metadata": metadata,
        "image": image,
        "slices": slices,
    }

In [31]:
from concurrent.futures import ThreadPoolExecutor


def generate_threads(worker, args=None, image_path=IMAGE_NAME):
    if args is None:
        args = []
    image_values = get_image_slice_values(image_path)
    metadata = image_values["metadata"]
    image = image_values["image"]
    slices = image_values["slices"]

    new_image = np.zeros_like(image)
    start_time = time.time()
    lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        futures = [
            executor.submit(worker, start, end, lock, new_image, image, *args)
            for start, end in slices
        ]

        for future in futures:
            future.result()

    elapsed = time.time() - start_time
    return new_image, image, metadata, elapsed


if __name__ == "__main__":
    new_image, image, metadata, elapsed = generate_threads(image_negative_worker)
    # After processing, result is your numpy array with altered bytes
    altered_bytes = new_image.tobytes()

    # Create PGM header
    header = f"P6\n{metadata.width} {metadata.height}\n255\n".encode()

    # Write to file
    with open("output.ppm", "wb") as f:
        f.write(header)
        f.write(altered_bytes)
    print(f"Time taken: {elapsed:.4f} seconds")

Time taken: 0.0103 seconds
